In [ ]:
import geopandas as gpd
from pathlib import Path
import fiona
import matplotlib.pyplot as plt

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
networks_folder = base_path / "Processed_data/networks"

# Define the output directory path
networks_catchments_intersections = base_path / "Processed_data/networks/networks_catchments_intersections"

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

In [ ]:
# Path to your water GeoPackage
waste_water_facilities_NWC_gpkg_path = networks_folder / "water/waste_water_facilities_NWC.gpkg"
waste_water_layers = fiona.listlayers(waste_water_facilities_NWC_gpkg_path)
print("Available layers:", waste_water_layers)

In [ ]:
hydrobasins = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
hydrobasins = gpd.read_file(hydrobasins)
print(hydrobasins.crs)

In [ ]:
# Read the roads layers (edges and nodes) from the GeoPackage.
wastewater_nodes = gpd.read_file(waste_water_facilities_NWC_gpkg_path, layer="nodes")

wastewater_nodes = wastewater_nodes.to_crs(jamaica_metric_grid_crs)

In [ ]:
wastewater_nodes.columns

In [ ]:
# === For Point Features (Wastewater Nodes) ===
# Perform a spatial join to attach hydrobasin attributes (e.g., HYBAS_ID) to each node.
wastewater_nodes_join = gpd.sjoin(wastewater_nodes, hydrobasins, how="left", predicate="intersects")

display(wastewater_nodes_join.head())
display(wastewater_nodes_join.columns)

# Example aggregation: Count the number of nodes per catchment
wastewater_nodes_count_by_catchment = wastewater_nodes_join.groupby("HYBAS_ID").size().reset_index(name="node_count")
display("\nNode Count by Catchment:")
display(wastewater_nodes_count_by_catchment)

In [ ]:
# Save the nodes join layer to a new GeoPackage
wastewater_nodes_catchments_intersection = networks_catchments_intersections / "wastewater_nodes_catchments_intersection.gpkg"
wastewater_nodes_join.to_file(wastewater_nodes_catchments_intersection, layer="joined_nodes", driver="GPKG")